# Build the RAG cache for SkinSight AI

This notebook generates the 35 cached strings the flashcard app loads at startup: 7 short summaries, 7 AI Explanations, and 21 feedback messages (1 correct-pick + 2 lookalike per condition).

Pipeline: chunk the 7 KB markdown files by section → embed with `sentence-transformers/all-MiniLM-L6-v2` → persist to a Chroma index → retrieve relevant chunks per condition → generate text with Ollama (`llama3.1:8b`, local) using four length-bounded prompt templates → save to `data/public/rag_cache.json`.

## Step 1 — Setup

Goal: verify Ollama is running and `llama3.1:8b` is pulled. Configure paths.

In [1]:
import json
import re
import shutil
from pathlib import Path

import ollama

try:
    ollama.show("llama3.1:8b")
    print("✓ llama3.1:8b is pulled and reachable")
except Exception as e:
    raise RuntimeError(f"Run `ollama pull llama3.1:8b` first.\n{e}")

ROOT = Path("..")
KB_DIR = ROOT / "data" / "public" / "kb"
CHROMA_DIR = ROOT / "data" / "public" / "kb_chroma"
CACHE_PATH = ROOT / "data" / "public" / "rag_cache.json"
TOP_PATH = ROOT / "data" / "public" / "top_conditions.json"
LL_PATH = ROOT / "data" / "public" / "lookalike_stats.json"

✓ llama3.1:8b is pulled and reachable


## Step 2 — Load and parse the 7 KB markdown files

Goal: read each card's markdown, split off the YAML frontmatter (condition slug, source URL) from the body. The body gets chunked next; the frontmatter becomes metadata on each chunk.

In [2]:
def parse_kb_file(path):
    text = path.read_text()
    m = re.match(r"^---\n(.*?)\n---\n(.*)$", text, re.DOTALL)
    if not m:
        raise ValueError(f"No frontmatter in {path.name}")
    fm_text, body = m.groups()
    fm = {}
    for line in fm_text.splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            fm[k.strip()] = v.strip()
    return fm, body

kb_files = sorted(f for f in KB_DIR.glob("*.md") if not f.name.startswith("_"))
print(f"Found {len(kb_files)} KB files\n")
for f in kb_files:
    fm, _ = parse_kb_file(f)
    print(f"  {fm['title']:42s}  ({fm['condition']})")

Found 7 KB files

  Basal cell carcinoma                        (basal_cell_carcinoma)
  Epidermal cyst                              (epidermal_cyst)
  Moles (melanocytic nevi, pigmented nevi)    (melanocytic_nevi)
  Mycosis fungoides                           (mycosis_fungoides)
  Seborrheic keratosis (brown warts, basal cell papillomas)  (seborrheic_keratosis)
  Squamous cell carcinoma in situ (intraepidermal squamous cell carcinoma)  (squamous_cell_carcinoma_in_situ)
  Viral wart — extra information              (verruca_vulgaris)


## Step 3 — Chunk each file by H2 section

Goal: split each KB body by `## Section` headings. Each chunk gets metadata (condition name, section title, source URL). Section-level chunks are coarse enough to read when debugging retrieval and fine enough for targeted retrieval per prompt.

In [3]:
from langchain_core.documents import Document

def split_by_h2(body):
    parts = re.split(r"^## ", body, flags=re.MULTILINE)
    chunks = []
    for part in parts[1:]:
        lines = part.split("\n", 1)
        section = lines[0].strip()
        content = lines[1].strip() if len(lines) > 1 else ""
        if content:
            chunks.append((section, content))
    return chunks

docs = []
for f in kb_files:
    fm, body = parse_kb_file(f)
    for section, content in split_by_h2(body):
        docs.append(Document(
            page_content=f"# {fm['title']} — {section}\n\n{content}",
            metadata={
                "condition_name": fm["title"],
                "condition_slug": fm["condition"],
                "section": section,
                "source": fm.get("source", ""),
                "url": fm.get("url", ""),
            },
        ))

print(f"Built {len(docs)} chunks from {len(kb_files)} files")
print(f"Sections seen: {sorted({d.metadata['section'] for d in docs})}")

Built 63 chunks from 7 files
Sections seen: ['Causes', 'Clinical features', 'Demographics', 'Diagnosis', 'Key distinguishing features', 'Overview', 'Risk factors and associations', 'Risk factors and prognosis', 'Treatment', 'Variation in skin tones']


## Step 4 — Build a persistent Chroma index

Goal: embed every chunk with `sentence-transformers/all-MiniLM-L6-v2` (384-dim, small, fast) and save the index to `data/public/kb_chroma/`. We persist so the prebuilt index ships with the deployment — no embedding work happens at runtime in the app.

We wipe `kb_chroma/` first so re-running this cell doesn't append duplicate chunks to an existing index.

In [4]:
#%pip install -q langchain-huggingface

In [5]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
CHROMA_DIR.mkdir(parents=True)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=str(CHROMA_DIR),
    collection_name="derm_kb",
)
print(f"Indexed {vectorstore._collection.count()} chunks at {CHROMA_DIR}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 63 chunks at ../data/public/kb_chroma


## Step 5 — Sanity-check retrieval

Goal: query the index for a phrase that should pull BCC chunks. If the wrong condition surfaces, embedding is broken — better to catch it before we waste 35 LLM calls.

In [6]:
hits = vectorstore.similarity_search(
    "pearly papule with arborizing telangiectasias and a rolled border",
    k=3,
)
for h in hits:
    print(f"[{h.metadata['condition_name']} / {h.metadata['section']}]")
    print(f"  {h.page_content[:140]}...\n")

[Basal cell carcinoma / Diagnosis]
  # Basal cell carcinoma — Diagnosis

Diagnosis starts clinically — the typical pearly papule with telangiectasias and a rolled border. Dermos...

[Viral wart — extra information / Overview]
  # Viral wart — extra information — Overview

Common warts (verruca vulgaris) appear as cauliflower-like papules with a rough, hyperkeratotic...

[Seborrheic keratosis (brown warts, basal cell papillomas) / Clinical features]
  # Seborrheic keratosis (brown warts, basal cell papillomas) — Clinical features

Seborrheic keratoses look like they're stuck on to the skin...



## Step 6 — Define the three prompt templates

Goal: one user prompt per output type, all sharing a common system instruction that enforces plain prose, length budget, and grounding to the retrieved context.

Length caps come from BOTH the prompt instruction (e.g. "max 60 words", "exactly THREE sentences") AND `num_predict` (a hard token cap on the model side) — belt and suspenders, since LLMs sometimes ignore prose-level instructions.

In [7]:
SYSTEM = (
    "You are a dermatology educator writing concise, accurate text for a "
    "medical-student flashcard app. Use only facts supported by the provided "
    "context. Write friendly, casual-but-clinical prose. "
    "STYLE RULES (every output): "
    "(a) ONE continuous paragraph — no line breaks, blank lines, numbering, "
    "bullets, headings, or preamble. "
    "(b) After first mention of the condition, use 'it' or 'they' (match "
    "number) — don't repeat the full name in every sentence. "
    "(c) Cut filler ('it is important to note', 'in general', 'when examining', "
    "'tends to'). Preserve clinical hedges ('may', 'can', 'usually') and exact "
    "descriptors ('vague', 'irregular', 'well-defined', 'atypical', 'pearly') "
    "— these change meaning. "
    "(d) Avoid 'characterized by' and 'distinguished by'. "
    "(e) Match verbs ('is/are', 'has/have') to the noun number. "
    "(f) Do not mention 'the student' or 'identification'."
)

PROMPT_SHORT = (
    "Write ONE descriptive sentence (max 20 words) about {name}: morphology "
    "plus the single most distinguishing feature. Start directly with an "
    "adjective or noun describing the lesion — do NOT start with the condition "
    "name, 'This is', or 'A/An'. "
    "Describe ONLY what a normal, healthy version of this condition looks "
    "like — do NOT include warning-sign features (irregular borders, "
    "asymmetry, rapid change, atypical cells) UNLESS {name} is one of "
    "{{Basal Cell Carcinoma, Squamous Cell Carcinoma In Situ, Mycosis "
    "Fungoides}}. "
    "Examples: 'Pearly papule with telangiectasias and a rolled border on "
    "sun-exposed skin.' / 'Stuck-on, waxy papules with a verrucous surface "
    "and sharp demarcation.'"
)

PROMPT_FEEDBACK_CORRECT = (
    "Confirm that {name} is the correct flashcard answer. Exactly THREE "
    "sentences (aim 35–40 words, max 45):\n"
    "(1) Open with 'Correct — ' (the word Correct + space + em-dash + space), "
    "then a plain-language definition of {name} with key clinical terms AND "
    "the most characteristic visual feature. Use '{name} is/are …', 'We can "
    "identify {name} by …', or 'This appears as …' after the em-dash. If "
    "{name} is one of {{Basal Cell Carcinoma, Squamous Cell Carcinoma In "
    "Situ, Mycosis Fungoides}}, add a brief stakes note here (e.g. 'a skin "
    "cancer worth catching'). Otherwise (for any benign condition), do NOT "
    "add a stakes note — do NOT invent precursor/progression warnings.\n"
    "(2) If the retrieved context mentions skin-tone variation for {name}, "
    "briefly note how appearance varies. Otherwise do NOT mention skin tones "
    "at all — do NOT write 'skin tone variation is not typically associated' "
    "or similar phrasing; simply omit the topic and continue to sentence 3.\n"
    "(3) The typical next step or treatment.\n"
    "Describe ONLY what a normal version of {name} looks like — do NOT "
    "include warning-sign features (irregular borders, asymmetry, rapid "
    "change, atypical cells) UNLESS {name} is one of the malignant/"
    "pre-invasive conditions listed above."
)

PROMPT_FEEDBACK_LOOKALIKE = (
    "The correct answer was {name} but the user picked {decoy}. Exactly "
    "THREE sentences (aim 35–40 words, max 45):\n"
    "(1) Open with 'This shows {name}.' or 'Close — this is {name}.' "
    "(vary between the two for tone). If {name} is one of {{Basal Cell "
    "Carcinoma, Squamous Cell Carcinoma In Situ, Mycosis Fungoides}}, add "
    "a brief stakes note here (e.g. 'a skin cancer worth catching'). "
    "Otherwise (for any benign condition), do NOT add a stakes note — do "
    "NOT invent precursor/progression warnings.\n"
    "(2) The bilateral contrast in ONE clean format: '{name} has [single "
    "feature] while {decoy} has [single feature]' (or 'looks like'). EXACTLY "
    "one distinguishing feature per side. Do NOT list multiple features.\n"
    "(3) If the retrieved context mentions skin-tone variation for {name}, "
    "briefly note it AND give the typical next step or treatment. Otherwise "
    "do NOT mention skin tones at all — just give the typical next step or "
    "treatment for {name}."
)

## Step 7 — Generate `short` for all 7 cards

Goal: for each condition, retrieve top-3 chunks from its own KB, format them as context for the SHORT prompt, and call Ollama. Save into a partial cache dict — we'll write to disk once at the end.

In [8]:
TOP_CONDS = json.loads(TOP_PATH.read_text())

def retrieve_for(name, k=4):
    """Top-k chunks from `name`'s KB (filtered by metadata)."""
    return vectorstore.similarity_search(
        f"clinical features, diagnosis, skin tone variation, and treatment of {name}",
        k=k,
        filter={"condition_name": name},
    )

def context_str(docs):
    return "\n\n".join(d.page_content for d in docs)

def generate(prompt, num_predict):
    resp = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": prompt},
        ],
        options={"temperature": 0.0, "num_predict": num_predict},
    )
    return resp["message"]["content"].strip()

cache = {name: {"definition": "", "feedback": {}} for name in TOP_CONDS}

for name in TOP_CONDS:
    ctx = context_str(retrieve_for(name, k=3))
    prompt = f"Context:\n{ctx}\n\n---\n{PROMPT_SHORT.format(name=name)}"
    cache[name]["definition"] = generate(prompt, num_predict=80)
    print(f"\n{name}:\n  {cache[name]['definition']}")


Melanocytic Nevi:
  Well-defined, brown to black macules or papules with a smooth surface and uniform pigmentation.

Seborrheic Keratosis:
  Well-defined, brown to tan, oval-shaped papules with a rough, verrucous surface and a distinct peripheral scale.

Verruca Vulgaris:
  Small, rough, hyperkeratotic papules with a central depression and a well-defined, verrucous surface.

Basal Cell Carcinoma:
  Well-defined, pearly papule with telangiectasias and a rolled border on sun-exposed skin.

Epidermal Cyst:
  Well-defined, smooth, dome-shaped papule with a thin epidermal covering and a small central punctum.

Mycosis Fungoides:
  Well-defined, erythematous patches with a characteristic "salmon-pink" hue and scaling or crusting in various stages of evolution.

Squamous Cell Carcinoma In Situ:
  Well-defined, flat-topped papules with a verrucous surface and scattered atypical keratinocytes.


## Step 9 — Generate feedback (correct + lookalike) for all 7 cards

Goal: 21 generations. Per condition we generate ONE correct-pick message (when the student answers right) and TWO lookalike messages (one per decoy in the MC quiz).

Lookalike pairs are computed from `lookalike_stats.json` — top-2 highest mean-cosine pairs per condition. This is the same source the DECK in `app.py` was built from, so the cache and the deck stay in sync automatically.

For lookalike feedback we retrieve from BOTH the correct condition (so the model knows what to point to) AND the lookalike (so the model knows what's actually being confused with what).

In [9]:
ll_stats = json.loads(LL_PATH.read_text())

LOOKALIKES = {}
for cond in TOP_CONDS:
    sims = sorted(
        ((other, s) for other, s in ll_stats["matrix"][cond].items() if other != cond),
        key=lambda kv: -kv[1],
    )
    LOOKALIKES[cond] = [other for other, _ in sims[:2]]

print("Lookalike pairs (computed from lookalike_stats.json):")
for cond, ll in LOOKALIKES.items():
    print(f"  {cond}: {ll}")
print()

for name in TOP_CONDS:
    ctx = context_str(retrieve_for(name, k=6))
    prompt = f"Context:\n{ctx}\n\n---\n{PROMPT_FEEDBACK_CORRECT.format(name=name)}"
    cache[name]["feedback"][name] = generate(prompt, num_predict=120)
    print(f"\n[{name}] correct-pick:")
    print(f"  {cache[name]['feedback'][name]}")

    for decoy in LOOKALIKES[name]:
        ctx_self = context_str(retrieve_for(name, k=6))
        ctx_decoy = context_str(retrieve_for(decoy, k=4))
        ctx = f"--- {name} context ---\n{ctx_self}\n\n--- {decoy} context ---\n{ctx_decoy}"
        prompt = (
            f"Context:\n{ctx}\n\n---\n"
            f"{PROMPT_FEEDBACK_LOOKALIKE.format(name=name, decoy=decoy)}"
        )
        cache[name]["feedback"][decoy] = generate(prompt, num_predict=120)
        print(f"\n[{name}] vs {decoy}:")
        print(f"  {cache[name]['feedback'][decoy]}")

Lookalike pairs (computed from lookalike_stats.json):
  Melanocytic Nevi: ['Seborrheic Keratosis', 'Verruca Vulgaris']
  Seborrheic Keratosis: ['Verruca Vulgaris', 'Squamous Cell Carcinoma In Situ']
  Verruca Vulgaris: ['Seborrheic Keratosis', 'Basal Cell Carcinoma']
  Basal Cell Carcinoma: ['Squamous Cell Carcinoma In Situ', 'Verruca Vulgaris']
  Epidermal Cyst: ['Seborrheic Keratosis', 'Verruca Vulgaris']
  Mycosis Fungoides: ['Verruca Vulgaris', 'Seborrheic Keratosis']
  Squamous Cell Carcinoma In Situ: ['Seborrheic Keratosis', 'Basal Cell Carcinoma']


[Melanocytic Nevi] correct-pick:
  Correct — Melanocytic Nevi are benign growths composed of melanocytes that can be found on sun-exposed areas and appear as well-defined, brown or black spots with a smooth surface and regular borders. We can identify Melanocytic Nevi by their characteristic "dotted" appearance due to the presence of small, pigmented nests within the epidermis. Typically, no treatment is necessary unless they become 

## Step 10 — Save the cache

Goal: write `rag_cache.json` to `data/public/`. The app loads this at startup and overrides the static DECK fields with these LLM-generated strings.

In [10]:
CACHE_PATH.write_text(json.dumps(cache, indent=2))

n_strings = sum(2 + len(v["feedback"]) for v in cache.values())
print(f"Wrote {CACHE_PATH.relative_to(ROOT)} ({CACHE_PATH.stat().st_size:,} bytes)")
print(f"Total cached strings: {n_strings}")

Wrote data/public/rag_cache.json (10,194 bytes)
Total cached strings: 35
